# Traditional Sentiment Analysis — Reddit Comments

Source:  (exploratory VADER section)

Lexicon-based sentiment analysis using VADER on Reddit election comments. Computes daily sentiment per candidate, sentiment difference (Trump − Harris), and volume-sentiment correlation.

**Data:** r/worldnews comments (exploratory; adapt file_path for other subreddits or use the full parquet from ).

In [ ]:
import nltk
nltk.download('vader_lexicon')

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

sentiment = sia.polarity_scores(body)
compound = sentiment["compound"]  # belangrijkste score (-1 tot 1)

In [ ]:
import json
import re
from collections import Counter, defaultdict
from nltk.sentiment import SentimentIntensityAnalyzer

# Exploratory: single subreddit file
# For full analysis, load from: ../../data/2_silver/reddit/reddit_full.parquet
file_path = "../../data/1_bronze/reddit/r_worldnews_comments.jsonl"

sia = SentimentIntensityAnalyzer()

daily_data = defaultdict(list)

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            comment = json.loads(line)
        except:
            continue

        body = comment.get("body", "")
        created_utc = comment.get("created_utc")

        if not isinstance(body, str):
            continue

        if body.lower() in ["[deleted]", "[removed]"]:
            continue

        # date (dag niveau)
        day = int(created_utc // 86400)  # unix → dag

        # clean text
        body_clean = body.lower()
        body_clean = re.sub(r"http\S+|www\S+", "", body_clean)

        # sentiment
        sentiment = sia.polarity_scores(body_clean)
        compound = sentiment["compound"]

        # candidate mentions
        trump = "trump" in body_clean
        harris = "harris" in body_clean or "kamala" in body_clean

        # save alles
        daily_data[day].append({
            "sentiment": compound,
            "trump": trump,
            "harris": harris,
            "length": len(body_clean)
        })

In [ ]:
import numpy as np

features = []

for day, comments in daily_data.items():

    sentiments = [c["sentiment"] for c in comments]
    trump_comments = [c for c in comments if c["trump"]]
    harris_comments = [c for c in comments if c["harris"]]

    feature_row = {
        "day": day,
        "n_comments": len(comments),

        # algemene sentiment
        "avg_sentiment": np.mean(sentiments),
        "std_sentiment": np.std(sentiments),

        # polariteit
        "pos_ratio": sum(s > 0.05 for s in sentiments) / len(sentiments),
        "neg_ratio": sum(s < -0.05 for s in sentiments) / len(sentiments),

        # candidate mentions
        "trump_mentions": len(trump_comments),
        "harris_mentions": len(harris_comments),

        # share of voice
        "trump_share": len(trump_comments) / len(comments),
        "harris_share": len(harris_comments) / len(comments),
    }

    # sentiment per kandidaat
    if trump_comments:
        feature_row["trump_sentiment"] = np.mean([c["sentiment"] for c in trump_comments])
    else:
        feature_row["trump_sentiment"] = 0

    if harris_comments:
        feature_row["harris_sentiment"] = np.mean([c["sentiment"] for c in harris_comments])
    else:
        feature_row["harris_sentiment"] = 0

    # verschil = super sterke feature
    feature_row["sentiment_diff"] = (
        feature_row["trump_sentiment"] - feature_row["harris_sentiment"]
    )

    features.append(feature_row)

In [ ]:
import pandas as pd

df = pd.DataFrame(features)

# sorteer op tijd
df = df.sort_values("day")

# maak echte datum (optioneel maar beter voor grafiek)
df["date"] = pd.to_datetime(df["day"], unit="D", origin="unix")

In [ ]:
df["trump_smooth"] = df["trump_sentiment"].rolling(7).mean()
df["harris_smooth"] = df["harris_sentiment"].rolling(7).mean()

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(df["date"], df["trump_smooth"], label="Trump sentiment")
plt.plot(df["date"], df["harris_smooth"], label="Harris sentiment")

plt.axhline(0, linestyle="--")

plt.title("Sentiment per kandidaat")
plt.xlabel("Datum")
plt.ylabel("Sentiment")
plt.legend()

plt.show()

In [ ]:
df["sent_diff_smooth"] = df["sentiment_diff"].rolling(7).mean()

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(df["date"], df["sent_diff_smooth"], label="Sentiment verschil (Trump - Harris)")

plt.axhline(0, linestyle="--")

plt.title("Sentiment voordeel Trump vs Harris")
plt.xlabel("Datum")
plt.ylabel("Sentiment verschil")

plt.legend()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14,6))

ax1.plot(df["date"], df["sentiment_smooth"], label="Sentiment", color="blue")
ax1.set_ylabel("Sentiment")

ax2 = ax1.twinx()
ax2.plot(df["date"], df["n_comments"], label="Aantal comments", color="orange", alpha=0.3)
ax2.set_ylabel("Volume")

plt.title("Sentiment vs volume")
fig.legend(loc="upper left")

plt.show()